# 1. Data preprocessing

This notebook aligns the four input datasets, applies sex-specific cohort-level scaling, creates the clinical benchmark clusters, and assigns the outer cross-validation folds.

## 1.1. Setup

In [ ]:
from pathlib import Path
import sys

project_dir = Path.cwd()
if not (project_dir / "src").exists():
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src import config
from src.data import (
    load_input_data,
    prepare_dataset,
    save_preprocessed_data,
    summarize_processed_data,
)
from src.training import set_deterministic_seed

set_deterministic_seed(config.RANDOM_STATE)

## 1.2. Load and prepare the data

In [ ]:
raw_data = load_input_data(
    config.GENOTYPE_PATH,
    config.PROTEOME_PATH,
    config.METABOLITE_PATH,
    config.CLINICAL_PATH,
)

processed_data = prepare_dataset(
    raw_data=raw_data,
    clinical_targets=config.CLINICAL_TARGETS,
    genotype_metadata_columns=config.GENOTYPE_METADATA_COLUMNS,
    id_columns={
        "genotype": config.GENOTYPE_ID_COLUMN,
        "proteome": config.OMICS_ID_COLUMN,
        "metabolite": config.OMICS_ID_COLUMN,
        "clinical": config.CLINICAL_ID_COLUMN,
    },
    sex_column=config.SEX_COLUMN,
    n_clusters=config.NUM_CLUSTERS,
    random_state=config.RANDOM_STATE,
    outer_splits=config.OUTER_SPLITS,
    outer_random_state=config.OUTER_RANDOM_STATE,
)

summary = summarize_processed_data(processed_data)
summary

## 1.3. Quality checks

In [ ]:
manifest = processed_data["participant_manifest"]
n_participants = len(manifest)

assert manifest["id"].is_unique
assert sorted(manifest["outer_fold"].unique()) == list(
    range(1, config.OUTER_SPLITS + 1)
)
assert processed_data["input_genotype"].shape[0] == n_participants
assert processed_data["input_proteome"].shape[0] == n_participants
assert processed_data["input_metabolite"].shape[0] == n_participants
assert processed_data["output_clinical"].shape == (
    n_participants, len(config.CLINICAL_TARGETS)
)

print("Participant alignment: OK")
print("Feature counts:", {
    "genotype": len(processed_data["genotype_features"]),
    "proteome": len(processed_data["proteome_features"]),
    "metabolite": len(processed_data["metabolite_features"]),
})
print("Benchmark counts:")
print(manifest["benchmark_cluster"].value_counts().sort_index())
print("Outer-fold counts:")
print(manifest["outer_fold"].value_counts().sort_index())

## 1.4. Save the processed dataset

In [ ]:
save_preprocessed_data(
    processed_data,
    output_dir=config.PROCESSED_DATA_DIR,
    dataset_path=config.PROCESSED_DATA_PATH,
)

print(f"Processed dataset: {config.PROCESSED_DATA_PATH}")
print(f"Participant manifest: {config.PARTICIPANT_MANIFEST_PATH}")